# Build a Multi-Agent Research Pipeline with LangGraph

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/langgraph/tutorial_research_agent.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Build a research agent that fans out parallel sub-topic researchers, each using LangGraph tool calling to search the web autonomously, then synthesizes all findings into a final report.

**Architecture:**
```
research_workflow (orchestrator)
  ├── plan_research → ["topic A", "topic B", "topic C"]
  ├── research_topic("topic A")  ┐
  ├── research_topic("topic B")  ├── parallel Flyte tasks
  ├── research_topic("topic C")  ┘
  └── synthesize_reports → final report
```

Each `research_topic` runs a LangGraph ReAct agent loop:
```
agent → (tool calls?) → tools → agent → (loop) → END
```

**Key concepts:**
- LangGraph `StateGraph` with `MessagesState` for agentic loops
- `ToolNode` + `bind_tools()` for LLM-driven tool calling
- Fan-out / fan-in parallelism with Flyte tasks
- Flyte reports for rich HTML output and graph visualization

---

## Setup

**Configuration:** Shared Flyte environment with Docker image + secrets. All LangGraph agents share this config.

In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/langgraph/
    !uv pip install -r requirements.txt

from utils.file_viewer import view_file

In [3]:
view_file("requirements.txt")

In [4]:
view_file("config.py")

## Connect to Flyte Cluster

Skip this step if you only want to run locally.

- Request demo access at [flyte.org](https://flyte.org/) if you don't have a cluster
- If you already have a cluster, set your endpoint below

In [ ]:
!flyte create config \
    --endpoint tryv2.hosted.unionai.cloud \
    --auth-type headless \
    --builder remote \
    --domain development \
    --project flytesnacks

In [5]:
view_file(".flyte/config.yaml")

## Set your API Keys

Create a `.env` file in the project root with your keys, or enter them below:

In [ ]:
# Skip if API keys are already set in .env or environment
import os
from getpass import getpass

os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')
os.environ['TAVILY_API_KEY'] = getpass('TAVILY_API_KEY: ')

To run on a remote Flyte cluster, add the API keys as secrets:

In [ ]:
!flyte create secret OPENAI_API_KEY
!flyte create secret TAVILY_API_KEY

## Run the Agent

Try running it before we walk through the code.

**Run locally:**

In [12]:
!python -m agent_research.workflow --query "Compare the top 6 programming languages for AI development: Python, Julia, R, Rust, Java, and Scala" --num-topics 6 --max-searches 2 --local 

Query: Compare the top 6 programming languages for AI development: Python, Julia, R, Rust, Java, and Scala
[d387d0e4-99b6-4cd7-a884-10f433a9e6dd][dufjpqqeccwrgwe2u49y6ynm1] Starting research workflow: Compare the top 6 programming languages for AI development: Python, Julia, R, Rust, Java, and Scala
[d387d0e4-99b6-4cd7-a884-10f433a9e6dd][cd0j06xds6qohgju2b8tlxr57] Planned 6 sub-topics: ['Overview and popularity of Python, Julia, R, Rust, Java, and Scala in AI development', 'Key features and language paradigms relevant to AI applications in each language', 'Performance benchmarks and efficiency considerations for AI workloads', 'Ecosystem, libraries, and frameworks supporting AI development in each language', 'Community support, documentation, and ease of learning for AI practitioners', 'Use cases and industry adoption trends of the six programming languages in AI projects']
[d387d0e4-99b6-4cd7-a884-10f433a9e6dd][6fzuo805cjhhcs46w3gp338o7] Researching: Overview and popularity of Python,

**Run on the remote Flyte cluster:**

The first run builds and pushes a container image, which may take some time.

In [13]:
!python -m agent_research.workflow --query "Compare the top 6 programming languages for AI development: Python, Julia, R, Rust, Java, and Scala" --num-topics 6 --max-searches 2

Query: Compare the top 6 programming languages for AI development: Python, Julia, R, Rust, Java, and Scala
08:19:09.597629 WARNING  remote_builder.py:95 -  Image                          
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-91dcd64ef1f0e83d5f59d0e1141c2b60 found. Skip    
                         building.                                              
08:19:09.602012 WARNING  _deploy.py:376 -  Built Image for environment          
                         langgraph_env, image:                                  
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-91dcd64ef1f0e83d5f59d0e1141c2b60                
Execution: rrgpwg4xd7zrwq2hmxnc | URL: https://demo.hosted.unionai.cloud/v2/domain/development/project/flytesnacks/runs/rrgpwg4xd7zrwq2hmxnc


---

# Code Walkthrough

The research agent has three layers:
1. **Tools** — Reusable search tool with Tavily
2. **Graph** — LangGraph agentic loop (agent → tools → agent)
3. **Workflow** — Flyte orchestrator (plan → parallel research → synthesize)

## 1. Web Search Tool

The search tool is a factory function that creates a LangChain `@tool` bound to a Tavily API key. The `@flyte.trace` decorator enables tool-level tracing in the Flyte UI.

In [8]:
view_file("tools/search.py")

**Key details:**
- `create_search_tool()` returns a tool closure bound to a specific API key
- The tool must be `async` to work with `@flyte.trace` and LangGraph's `ToolNode`
- `@tool` decorator makes it compatible with LangChain's tool calling interface
- The LLM sees the docstring as the tool description and decides when/how to call it

## 2. LangGraph Research Graph

The graph is a simple agentic loop: the LLM decides whether to search or produce a final answer.

```
START → agent → should_continue?
                   ├─ has tool_calls → tools → agent (loop)
                   └─ no tool_calls → END
```

In [9]:
view_file("agent_research/graph.py")

**Key details:**
- `MessagesState` — LangGraph's built-in state that maintains a list of messages (conversation history)
- `bind_tools(tools)` — Tells the LLM about available tools so it can generate tool calls
- `ToolNode(tools)` — LangGraph's built-in node that executes tool calls from the LLM response
- `should_continue` — Conditional edge that checks if the LLM wants to call more tools or is done
- The system prompt tells the agent how many searches to use and what kind of output to produce
- `graph.compile()` returns a runnable that can be invoked with `await graph.ainvoke(...)`

## 3. Flyte Workflow

The workflow orchestrates three Flyte tasks:

1. **`plan_research`** — LLM breaks the query into sub-topics
2. **`research_topic`** — Runs the LangGraph graph on each sub-topic (parallel via `asyncio.gather`)
3. **`synthesize_reports`** — LLM combines all sub-topic reports into a final report

Each task shows up separately in the Flyte UI with its own report and trace.

In [10]:
view_file("agent_research/workflow.py")

**Key details:**
- `report=True` on `@env.task` enables Flyte reports — rich HTML output visible in the Flyte UI
- `flyte.report.replace.aio()` updates the report content (replaces previous content)
- `flyte.report.get_tab()` creates named tabs for sub-topic reports in the synthesizer
- `draw_mermaid_png()` generates a graph visualization embedded as a base64 image
- All tasks return `str` (JSON serialized) for proper display in the Flyte UI
- The orchestrator fans out research tasks with `asyncio.gather` for parallel execution

---

## More Examples

Try different research queries:

In [ ]:
!python -m agent_research.workflow --local --query "What are the pros and cons of microservices vs monoliths?"

In [ ]:
!python -m agent_research.workflow --local --query "Latest developments in large language models" --num-topics 4 --max-searches 3

**CLI flags:**

| Flag | Default | Description |
|------|---------|-------------|
| `--local` | off | Run locally instead of on Flyte cluster |
| `--query` | required | Research question |
| `--num-topics` | 3 | Number of sub-topics to research in parallel |
| `--max-searches` | 2 | Max web searches per sub-topic |

---

## Key Takeaways

| Concept | What it does |
|---------|-------------|
| `StateGraph` + `MessagesState` | Manages conversation history in the agentic loop |
| `bind_tools()` + `ToolNode` | LLM-driven tool calling — the model decides when to search |
| Conditional edges | Route between "keep searching" and "done" based on LLM output |
| `@env.task(report=True)` | Each Flyte task gets its own report, trace, and UI entry |
| `asyncio.gather` | Fan-out parallel research across sub-topics |
| `@flyte.trace` | Tool-level tracing visible in the Flyte UI |

**Other LangGraph agent examples in this project:**
- [ReAct Agent](tutorial_react_agent.ipynb) — Reason → Act → Observe with math and string tools
- [Reflection Agent](tutorial_reflection_agent.ipynb) — Generate → Critique → Refine loop

---

## Resources

- Full code: `tutorials/langgraph/`
- [LangGraph docs](https://langchain-ai.github.io/langgraph/)
- [Flyte docs](https://docs.flyte.org)
- Questions? Join the Flyte community Slack!